# Medical LLM Fine-Tuning: QLoRA Training with Mistral-7B-Instruct

This notebook performs **QLoRA (Quantized Low-Rank Adaptation)** fine-tuning of `mistralai/Mistral-7B-Instruct-v0.2` on the cleaned medical Q&A dataset produced in notebooks 01 and 02. The fine-tuned adapter will be pushed to HuggingFace Hub under `labdhu/mistral-7b-medical-qa`.

> **Environment:** Google Colab — **T4 GPU runtime required.** Go to `Runtime > Change runtime type > T4 GPU` before running any cell.

## Cell 1 — Install Dependencies

We install all libraries required for the fine-tuning pipeline:
- **`peft`** — HuggingFace's Parameter-Efficient Fine-Tuning library. Provides `LoraConfig` and `get_peft_model` to attach lightweight LoRA adapters to the frozen base model.
- **`trl`** — Transformer Reinforcement Learning library. Provides `SFTTrainer` (Supervised Fine-Tuning Trainer), which handles the training loop, evaluation, and checkpointing automatically.
- **`bitsandbytes`** — Enables 4-bit NF4 quantization via `BitsAndBytesConfig`, reducing the 7B model from ~14GB to ~4GB VRAM.
- **`accelerate`** — Powers `device_map="auto"` for intelligent layer placement across CPU and GPU.
- **`wandb`** — Weights & Biases experiment tracker. Streams loss curves, hyperparameters, and evaluation metrics to a live dashboard.
- **`huggingface_hub`** — Provides `login()` and `push_to_hub()` for authenticating and uploading the trained adapter.

In [1]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate wandb huggingface_hub sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.3 MB/s eta 0:00:00


## Cell 2 — Mount Google Drive, Authenticate HuggingFace & Initialise W&B

Three setup tasks happen here:

1. **Google Drive mount** — Colab's local filesystem is wiped on every disconnect. Mounting Drive gives us a persistent `PROJECT_DIR` where all training checkpoints and adapter weights are saved safely, even if the runtime crashes mid-training.

2. **HuggingFace authentication** — Mistral-7B-Instruct is a gated model requiring a HuggingFace account token with `read` access. We also need a token with `write` access to push the fine-tuned adapter to Hub at the end. The token is read from Colab's `Secrets` panel (`HF_TOKEN`) — never hard-coded.

3. **W&B initialisation** — `wandb.init()` starts an experiment run and registers all training hyperparameters upfront. Every `trainer.log()` call made internally by `SFTTrainer` will stream metrics (loss, learning rate, evaluation loss) to the W&B dashboard in real time, giving full training visibility without any extra code.

In [2]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
import wandb
from huggingface_hub import login

# Set environment variable to potentially mitigate CUDA OOM due to fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Persistent project directory — all outputs land here
PROJECT_DIR = '/content/drive/MyDrive/medical-llm'
os.makedirs(PROJECT_DIR, exist_ok=True)

# Authenticate with HuggingFace Hub (token stored in Colab Secrets)
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("HuggingFace login successful")

# Authenticate and initialise W&B experiment tracking
WANDB_API_KEY = userdata.get('WANDB_API_KEY')
wandb.login(key=WANDB_API_KEY)

wandb.init(
    project="medical-llm-finetune",
    name="mistral-7b-qlora-run-2",
    config={
        "model": "mistralai/Mistral-7B-Instruct-v0.2",
        "dataset": "lavita/medical-qa-datasets all-processed",
        "train_samples": 8000,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "learning_rate": 2e-4,
        "batch_size": 4,
        "epochs": 1,
        "max_seq_length": 512
    }
)
print("W&B initialized successfully")

Mounted at /content/drive
HuggingFace login successful


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: labdhu (labdhu-toss-solutions) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B initialized successfully


## Cell 3 — Load and Format Training Data

We load the pre-split `train_data.csv` and `val_data.csv` files that were saved to Google Drive in notebook 02. Each sample has three fields: `instruction`, `input`, and `output`.

**Why prompt formatting matters:** Mistral-7B-Instruct was pre-trained using a specific chat template with `[INST]` and `[/INST]` delimiters. During fine-tuning, every training sample must follow this **exact** format:
```
<s>[INST] {instruction}\n\n{input} [/INST] {output} </s>
```
Deviating from this template — even slightly — causes the model to see training data as out-of-distribution, leading to poor convergence and incoherent outputs. The `format_mistral_prompt` function applies this transformation to every row, and we use HuggingFace `Dataset.map()` to apply it efficiently across all 15,200 samples.

In [3]:
import pandas as pd
from datasets import Dataset

# Load pre-split CSVs from Google Drive (written by notebook 02)
train_df = pd.read_csv(f'{PROJECT_DIR}/train_data.csv')
train_df = train_df.iloc[:8000].reset_index(drop=True)
val_df   = pd.read_csv(f'{PROJECT_DIR}/val_data.csv')

print(f"Train samples: {len(train_df)}")
print(f"Val samples:   {len(val_df)}")

def format_mistral_prompt(row):
    """Format a dataset row into Mistral-7B-Instruct's exact chat template."""
    return {
        "text": f"<s>[INST] {row['instruction']}\n\n{row['input']} [/INST] {row['output']} </s>"
    }

# Convert pandas DataFrames to HuggingFace Dataset and apply the prompt template
train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(format_mistral_prompt)
val_dataset   = val_dataset.map(format_mistral_prompt)

print("\nSample formatted training example:")
print(train_dataset[0]['text'][:300])
print("...")
print(f"\nTotal characters in example: {len(train_dataset[0]['text'])}")

Train samples: 8000
Val samples:   200


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]


Sample formatted training example:
<s>[INST] Answer the medical question posed in the input.

What is the historical background of liposarcoma? [/INST] Liposarcoma was first discovered in 1857 by Rudolph Virchow, a German pathologist, who described a tumor arising from fat tissue. Originally, Virchow called the tumor "myxoma lipomato
...

Total characters in example: 1352


## Cell 4 — Load Mistral-7B in 4-Bit and Configure LoRA Adapters

### 4-Bit NF4 Quantization
The full Mistral-7B model requires ~14GB VRAM in BF16 — more than the T4's 15GB budget once training overhead is factored in. `BitsAndBytesConfig` with `load_in_4bit=True` compresses model weights to NormalFloat4 format, reducing footprint to ~4GB while keeping computation in BF16 for numerical stability.

### LoRA: Why it works
Standard full fine-tuning updates all 7 billion parameters — impossible on a free GPU. LoRA (Low-Rank Adaptation) instead **freezes every base model weight** and injects small trainable rank-decomposition matrices alongside the attention projections. The number of trainable parameters drops from 7B to ~21M — less than **0.3% of the total**.

### LoRA Hyperparameter Rationale
| Parameter | Value | Reason |
|---|---|---|
| `r` | 16 | Adapter rank — controls capacity. 16 is the standard starting point for domain adaptation. |
| `lora_alpha` | 32 | Scaling factor = 2× rank. This ratio keeps gradient magnitudes stable during training. |
| `target_modules` | `q_proj, v_proj, k_proj, o_proj` | All four attention projections — gives maximum expressiveness without touching FFN layers. |
| `lora_dropout` | 0.05 | Light dropout to prevent overfitting on our 15k sample dataset. |
| `bias` | `none` | Bias terms are not adapted — standard for QLoRA. |

In [4]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Clear any leftover memory before loading
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

free_mem, total_mem = torch.cuda.mem_get_info()
print(f"GPU memory before loading: {free_mem/1e9:.1f} GB free / {total_mem/1e9:.1f} GB total")

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit NF4 quantization — reduces model from 14GB to ~4GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_nested_quant=False
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=HF_TOKEN,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model in 4-bit... (3-5 minutes)")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
    token=HF_TOKEN,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

free_mem, total_mem = torch.cuda.mem_get_info()
print(f"GPU memory after loading: {free_mem/1e9:.1f} GB free / {total_mem/1e9:.1f} GB total")

# Prepare model for k-bit training
model.config.use_cache = False
model.config.pretraining_tp = 1
model = prepare_model_for_kbit_training(model)

# LoRA adapter configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Total parameters:     {total_params:,}")
print(f"Percentage trained:   {100 * trainable_params / total_params:.4f}%")
print(f"\nModel device: {next(model.parameters()).device}")

GPU memory before loading: 15.5 GB free / 15.6 GB total
Loading tokenizer...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

GPU memory after loading: 11.0 GB free / 15.6 GB total

Trainable parameters: 13,631,488
Total parameters:     3,765,702,656
Percentage trained:   0.3620%

Model device: cuda:0


## Cell 5 — Configure the SFTTrainer

`SFTTrainer` from TRL wraps HuggingFace's `Trainer` with supervised fine-tuning utilities tailored for language models. It handles the training loop, gradient accumulation, evaluation scheduling, checkpoint saving, and metric reporting automatically.

### Key Training Argument Decisions

| Argument | Value | Rationale |
|---|---|---|
| `per_device_train_batch_size` | 4 | Maximum that fits in T4 VRAM with 4-bit model + activations. |
| `gradient_accumulation_steps` | 4 | Simulates effective batch size of **16** without extra memory — gradients are accumulated over 4 forward passes before one weight update. |
| `fp16` | `True` | Half-precision arithmetic halves activation memory and speeds up matrix multiplications on T4 tensor cores. |
| `optim` | `paged_adamw_8bit` | Memory-efficient 8-bit AdamW optimizer specifically designed for QLoRA. Pages optimizer states to CPU RAM when not needed, preventing OOM. |
| `lr_scheduler_type` | `cosine` | Cosine decay smoothly reduces the learning rate, avoiding sharp loss spikes at the end of training. |
| `eval_steps` | 100 | Evaluates on the 200-sample validation set every 100 training steps — early warning signal for overfitting. |
| `load_best_model_at_end` | `True` | Automatically restores the checkpoint with the lowest validation loss before saving — prevents saving an overfit model. |
| `report_to` | `wandb` | Streams all `trainer.log()` calls to W&B — no extra code needed for metric tracking. |

In [13]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=f"{PROJECT_DIR}/checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=25,
    eval_steps=100,
    save_steps=200,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="wandb",
    run_name="mistral-7b-qlora-run-2",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    dataset_text_field="text",
    max_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

effective_batch = sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps
total_steps = len(train_dataset) // effective_batch

print("Trainer configured successfully")
print(f"Effective batch size: {effective_batch}")
print(f"Total training steps: {total_steps}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Trainer configured successfully
Effective batch size: 16
Total training steps: 500


## Cell 6 — Run QLoRA Training

This cell executes the full training run. `trainer.train()` orchestrates everything:
- **Forward pass** through the quantized model + LoRA adapters
- **Loss calculation** using cross-entropy on the target tokens
- **Backward pass** computing gradients only through the adapter parameters (base model is frozen)
- **Gradient accumulation** over 4 steps before each weight update
- **Checkpointing** every 200 steps to Google Drive
- **Evaluation** on the 200-sample validation set every 100 steps

**Expected duration:** ~2 hours on a free Colab T4 GPU for 1 epoch over 15,000 samples.

**Healthy loss curve indicators:**
| Signal | Interpretation |
|---|---|
| Loss starts ~2.0, converges to 0.7–0.9 | ✅ Normal healthy training |
| Loss stays above 1.5 throughout | ⚠️ Learning rate may be too low |
| Loss drops below 0.4 before step 500 | ⚠️ Possible overfitting — check val loss |
| Train loss ↓ but val loss ↑ | ❌ Overfitting — stop early |

All metrics are streaming live to the W&B dashboard at [wandb.ai](https://wandb.ai).

In [14]:
# Reset model params back to their natural dtype before training
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

print("Trainable params cast to float32 for stable gradient computation")
print(f"Sample param dtype: {next(p for p in model.parameters() if p.requires_grad).dtype}")

print("Starting QLoRA fine-tuning...")
print("Monitor live at: https://wandb.ai")
print("-" * 50)

trainer.train()

print("-" * 50)
print("Training complete")

wandb.log({
    "final_train_loss": trainer.state.log_history[-1].get("train_loss", 0),
    "total_steps": trainer.state.global_step
})

wandb.finish()
print("W&B run finished")

Trainable params cast to float32 for stable gradient computation
Sample param dtype: torch.float32
Starting QLoRA fine-tuning...
Monitor live at: https://wandb.ai
--------------------------------------------------


Step,Training Loss,Validation Loss
100,1.878204,1.877294
200,1.907191,1.861634
300,1.911198,1.855905
400,1.887766,1.849214
500,1.873566,1.845777


--------------------------------------------------
Training complete


eval/entropy,▂█▁▂▂
eval/loss,█▅▃▂▁
eval/mean_token_accuracy,▁▅▆██
eval/num_tokens,▁▃▄▆█
eval/runtime,█▁▁▆▅
eval/samples_per_second,▁█▇▃▄
eval/steps_per_second,▁██▁▃
final_train_loss,▁
total_steps,▁
train/entropy,▄██▅▅▅▂▅▇▄▁▅▃▅▂▄▅▅▂▂
+7,...


W&B run finished


## Cell 7 — Save Adapter Weights and Push to HuggingFace Hub

### What gets saved
We save **only the LoRA adapter weights** — not the full 7B base model. The adapter is a small collection of low-rank matrices (typically 50–150MB) that encode everything the model learned during fine-tuning. The full Mistral-7B-Instruct base weights remain unchanged and stay on HuggingFace Hub.

This is standard practice for sharing LoRA fine-tuned models:
- **Storage:** Adapter is ~100MB vs ~14GB for the full model.
- **Reproducibility:** Anyone can reconstruct the full fine-tuned model by loading `mistralai/Mistral-7B-Instruct-v0.2` and calling `PeftModel.from_pretrained(base_model, adapter_repo)`.
- **Composability:** The adapter can be merged or swapped without touching the base.

### Two-step save strategy
1. **Local Drive save** — Immediate backup in case the Colab session ends before the Hub push completes.
2. **HuggingFace Hub push** — Publishes the adapter to `labdhu/mistral-7b-medical-qa` for permanent storage, sharing, and use in notebook 04 evaluation.

In [15]:
# Step 1: Save adapter weights locally to Google Drive as a safety backup
adapter_save_path = f"{PROJECT_DIR}/mistral-medical-adapter"
model.save_pretrained(adapter_save_path)
tokenizer.save_pretrained(adapter_save_path)
print(f"Adapter weights saved locally to: {adapter_save_path}")

# Step 2: Push adapter to HuggingFace Hub for permanent storage and sharing
REPO_ID = "labdhu/mistral-7b-medical-qa"

print(f"Pushing to HuggingFace Hub: {REPO_ID}")
model.push_to_hub(REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)

print(f"Model successfully pushed to: https://huggingface.co/{REPO_ID}")
print("Week 2 complete.")

Adapter weights saved locally to: /content/drive/MyDrive/medical-llm/mistral-medical-adapter
Pushing to HuggingFace Hub: labdhu/mistral-7b-medical-qa


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|1         |  558kB / 54.6MB            

README.md: 0.00B [00:00, ?B/s]

Model successfully pushed to: https://huggingface.co/labdhu/mistral-7b-medical-qa
Week 2 complete.
